# 02_GSE138720_qc_selection

**Thesis Methods section(s): 4.1.1, 4.1.2, 4.1.3**

**Reads:** GSE138720 expression matrices and metadata (GEO).

**Writes:** GSE138720_CD8_STRICT_postQC_scvi_rawcounts.h5ad

**Notes:** Cells were already CD8-restricted by the original authors; no marker rule applied. This notebook also contains an exploratory per-dataset scVI model that is NOT part of the pipeline reported in the thesis (see README).

Input data are not included in this repository. Set `DATA_ROOT` below to a local folder
holding the GEO downloads; see `README.md` for accessions and the expected layout.


In [ ]:
# Root folder for input data (NOT included in this repository).
# Set the DATA_ROOT environment variable, or edit the fallback below.
import os
DATA_ROOT = os.environ.get("DATA_ROOT", "data")


In [ ]:
# Load necessary libraries
import os
import scanpy as sc
import scvi
import matplotlib.pyplot as plt

In [ ]:
# 📁 Path to the parent directory containing all patient folders
parent_dir = f"{DATA_ROOT}/Single-cell RNA seq/Collaboration with FB/GSE138720 - Melanoma/Data"  # 🔁 CHANGE this to your actual path

In [ ]:
# 🧠 Define function to load each folder as AnnData
def load_10x_folder(folder_name):
    path = os.path.join(parent_dir, folder_name) # Create the full path to the folder
    adata = sc.read_10x_mtx(path, var_names="gene_symbols", cache=True) # Load the single-cell data
    adata.obs["sample_id"] = folder_name # Add a column to store which patient the data is from
    if "Pt" in folder_name:
        patient_id = folder_name.split("_")[-1]
        adata.obs["patient_id"] = patient_id # Extract patient ID from the folder name
    if "TIL" in folder_name: # If the folder name contains "TIL"
        adata.obs["source"] = "TIL" # Mark the source as Tumor-Infiltrating Lymphocytes
    elif "PBL" in folder_name: # If the folder name contains "PBL"
        adata.obs["source"] = "PBL" # Mark the source as Peripheral Blood Lymphocytes
    return adata # Return the loaded data

In [ ]:
# 📂 All folder names
folders = [
    "CD8PBL_Pt1", "CD8PBL_Pt2", "CD8PBL_Pt3", "CD8PBL_Pt4",
    "CD8PBL_Pt5", "CD8PBL_Pt6", "CD8PBL_Pt7", "CD8PBL_Pt8",
    "CD8TIL_Pt1", "CD8TIL_Pt2", "CD8TIL_Pt3", "CD8TIL_Pt4",
    "CD8TIL_Pt5", "CD8TIL_Pt6", "CD8TIL_Pt7", "CD8TIL_Pt8"
] # 8 PBL and 8 TIL folders

# 🧬 Load and store all AnnData objects in a list
all_adata = [load_10x_folder(f) for f in folders] # This gives you 16 AnnData objects

# 🔗 Merge all into one AnnData object
GSE138720 = all_adata[0].concatenate(*all_adata[1:], batch_key="sample_id", batch_categories=folders)

# ✅ Now adata.obs contains:
# - `adata.obs["patient"]` tells you which patient each cell came from
# - `adata.obs["source"]` tells you whether it came from blood (PBL) or tumor (TIL)

In [ ]:
GSE138720.obs.columns

In [ ]:
GSE138720.obs["sample_id"].value_counts()

In [ ]:
GSE138720.obs["patient_id"].value_counts()

In [ ]:
GSE138720.obs["source"].value_counts()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 📐 Count cells per patient and source
counts = GSE138720.obs.groupby(["patient_id", "source"]).size().reset_index(name="cell_count")

# 🎨 Set the style
sns.set(style="whitegrid")

# 📊 Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=counts, x="patient_id", y="cell_count", hue="source", palette="viridis")

# ✨ Decorations
plt.title("Cell Counts per Patient and Sample Type (TIL vs PBL)", fontsize=14)
plt.xlabel("Patient ID", fontsize=12)
plt.ylabel("Number of Cells", fontsize=12)
plt.legend(title="Sample Source")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Quality Control (QC)
import scanpy as sc
import matplotlib.pyplot as plt

# Calculate QC metrics
GSE138720.var['mt'] = GSE138720.var_names.str.upper().str.startswith('MT-')  # Ensures uppercase compatibility
sc.pp.calculate_qc_metrics(GSE138720, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# Plot QC metrics before filtering
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

# Total counts per cell
axs[0].hist(GSE138720.obs['total_counts'], bins=50, color='skyblue')
axs[0].set_title('Total Counts per Cell')
axs[0].set_xlabel('Total Counts')
axs[0].set_ylabel('Number of Cells')

# Number of genes per cell
axs[1].hist(GSE138720.obs['n_genes_by_counts'], bins=50, color='lightgreen')
axs[1].set_title('Number of Genes per Cell')
axs[1].set_xlabel('Genes')
axs[1].set_ylabel('Number of Cells')

# Percent mitochondrial genes
axs[2].hist(GSE138720.obs['pct_counts_mt'], bins=50, color='salmon')
axs[2].set_title('% Mitochondrial Genes')
axs[2].set_xlabel('% MT Genes')
axs[2].set_ylabel('Number of Cells')

plt.tight_layout()
plt.show()

In [ ]:
# You can now visually decide thresholds.
# Here's one possible filtering step based on common practice:
sc.pp.filter_cells(GSE138720, min_genes=200)
sc.pp.filter_genes(GSE138720, min_cells=3)
GSE138720 = GSE138720[GSE138720.obs.pct_counts_mt < 10, :]  # Modify threshold based on histogram

In [ ]:
# 📌 Save raw version for scVI
GSE138720_raw_scVI = GSE138720.copy()

In [ ]:
# 📍 UMAP before scVI (PCA-based)

sc.pp.normalize_total(GSE138720, target_sum=1e4)
sc.pp.log1p(GSE138720)
sc.pp.highly_variable_genes(GSE138720, n_top_genes=2000, subset=True)
sc.pp.scale(GSE138720, max_value=10)
sc.tl.pca(GSE138720, svd_solver='arpack')

In [ ]:
# 🔍 Elbow plot to choose number of PCs
import matplotlib.pyplot as plt
import numpy as np

explained = GSE138720.uns["pca"]["variance_ratio"]
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(explained) + 1), explained, marker='o')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Elbow Plot')
plt.grid(True)
plt.show()

In [ ]:
# Dimensionality reduction & clustering
sc.pp.neighbors(GSE138720, n_neighbors=15, n_pcs=9)
sc.tl.umap(GSE138720)
sc.tl.leiden(GSE138720, resolution=0.5)

# UMAP plot (PCA-based)
sc.pl.umap(
    GSE138720,
    color=["leiden"],
    title=["Before scVI: Clusters"]
)

In [ ]:
# UMAP plot (PCA-based)
sc.pl.umap(
    GSE138720,
    color=["sample_id"],
    title=["Before scVI: Sample ID"],
)

In [ ]:
# UMAP plot (PCA-based)
sc.pl.umap(
    GSE138720,
    color=["patient_id"],
    title=["Before scVI: Patient ID"],
)

In [ ]:
# UMAP plot (PCA-based)
sc.pl.umap(
    GSE138720,
    color=["source"],
    title=["Before scVI: Source (PBL vs TIL)"],
)

In [ ]:
# Optionally save the file
GSE138720.write(
    f"{DATA_ROOT}/scVI/GSE138720_PCA_based.h5ad"
)

In [ ]:
# Load the saved AnnData object
GSE138720 = sc.read(f"{DATA_ROOT}/scVI/GSE138720_PCA-based.h5ad")

In [ ]:
# Optionally save the file
GSE138720_raw_scVI.write(
    f"{DATA_ROOT}/scVI/GSE138720_raw.h5ad"
)

In [ ]:
# Load the saved AnnData object
GSE138720_raw_scVI = sc.read(f"{DATA_ROOT}/scVI/GSE138720_raw.h5ad")

In [ ]:
import os
import scvi

# Define save path
model_path = "scvi_model/GSE138720_scvi_model"

# Make sure the folder exists
os.makedirs("scvi_model", exist_ok=True)

# Set up AnnData for scVI
scvi.model.SCVI.setup_anndata(GSE138720_raw_scVI, batch_key="sample_id")

# Load or train
if os.path.exists(os.path.join(model_path, "model.pt")):
    model = scvi.model.SCVI.load(model_path, GSE138720_raw_scVI)
    print("✅ Loaded saved scVI model.")
else:
    model = scvi.model.SCVI(GSE138720_raw_scVI)
    model.train(max_epochs=70)
    model.save(model_path, overwrite=True)
    print("✅ Trained and saved scVI model.")

In [ ]:
# Plot scVI Training Loss Curve
# ----------------------------------
import matplotlib.pyplot as plt

# Get history directly from trained model
history = model.history
print("Available keys in history:", history.keys())
elbo = history["elbo_train"]

# Plot
plt.plot(range(1, len(elbo) + 1), elbo, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Training ELBO Loss")
plt.title("scVI Training Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Get scVI latent space and compute UMAP
GSE138720_raw_scVI.obsm["X_scVI"] = model.get_latent_representation()

In [ ]:
# Optionally save the file
GSE138720_scVI_based.write(
    f"{DATA_ROOT}/scVI/GSE138720_scVI_based.h5ad"
)

In [ ]:
# Load the saved AnnData object
GSE138720_scVI_based = sc.read(f"{DATA_ROOT}/scVI/GSE138720_scVI_based.h5ad")

In [ ]:
# UMAP & Clustering on scVI Latent Space
# --------------------------------------

# Step 1: Compute neighborhood graph using the scVI latent space
sc.pp.neighbors(GSE138720_scVI_based, use_rep="X_scVI", n_neighbors=15)

# Step 2: Run UMAP (2D)
sc.tl.umap(GSE138720_scVI_based, n_components=2)

# Step 3: Leiden clustering
sc.tl.leiden(GSE138720_scVI_based, resolution=0.5)

# Step 4 (Optional): Check number of clusters
n_clusters = GSE138720_scVI_based.obs["leiden"].nunique()
print(f"✅ Leiden clustering at resolution 0.5 → {n_clusters} clusters")

Export the scVI-processed object for Seurat: expression matrix, cell metadata, scVI latent space and UMAP coordinates.

In [ ]:
# UMAP plot
sc.pl.umap(
    GSE138720_scVI_based,
    color=["leiden"],
    title=["After scVI: Clusters"],
)

In [ ]:
# UMAP plot
sc.pl.umap(
    GSE138720_scVI_based,
    color=["patient_id"],
    title=["After scVI: Patient ID"],
)

In [ ]:
# UMAP plot
sc.pl.umap(
    GSE138720_raw_scVI,
    color=["source"],
    title=["After scVI: Source (PBL vs TIL)"],
)

In [ ]:
# UMAP plot
sc.pl.umap(
    GSE138720_raw_scVI,
    color=["sample_id"],
    title=["After scVI: Sample ID"],
)

In [ ]:
# Load the scVI-processed data
GSE138720_scVI = sc.read(f"{DATA_ROOT}/scVI/2 GSE138720/2C GSE138720_scVI_based.h5ad")

In [ ]:
# Dataset subset: GSE108989
GSE138720_scVI = GSE138720_scVI[GSE138720_scVI.obs["cell_type"] == "CD8", :]  

In [ ]:
GSE138720_scVI.obs.columns

In [ ]:
GSE138720_scVI.obs["tissue_group"].value_counts()

In [ ]:
# Rename 'old_col' -> 'new_col'
GSE138720_scVI.obs.rename(columns={'source': 'tissue_group'}, inplace=True)

In [ ]:
# mapping
rename_map = {"PBL": "Blood", "TIL": "Tumor"}

# If 'source' is categorical:
if str(GSE138720_scVI.obs["source"].dtype) == "category":
    GSE138720_scVI.obs["source"] = (
        GSE138720_scVI.obs["source"]
        .cat.rename_categories(rename_map)
        .cat.set_categories(["Blood", "Tumor"])   # optional: set order
    )
else:
    # plain string column
    GSE138720_scVI.obs["source"] = (
        GSE138720_scVI.obs["source"].replace(rename_map)
    )

# quick check
print(GSE138720_scVI.obs["source"].value_counts())

In [ ]:
# Load necessary libraries
from scipy.io import mmwrite
from scipy.sparse import csr_matrix
import scanpy as sc
import pandas as pd
import numpy as np

adata = GSE138720_scVI.copy()

In [ ]:
# 1) raw counts in .X
if "counts" in adata.layers:
    adata.X = adata.layers["counts"]

In [ ]:
# 2) basic clean-up
sc.pp.filter_genes(adata, min_counts=1)
adata.obs_names_make_unique()
adata.var_names_make_unique()
adata.var_names = adata.var_names.str.replace("_","-", regex=False)

In [ ]:
# 3) compact storage
adata.X = csr_matrix(adata.X).astype(np.int32)

In [ ]:
# --- Sanity: shapes you'll export ---
n_cells, n_genes = adata.n_obs, adata.n_vars
print(f"Cells: {n_cells:,}   Genes: {n_genes:,}")

In [ ]:
# 4) Write MTX **transposed** = genes x cells (what Seurat expects)
mmwrite("counts.mtx", adata.X.T)   # <-- key change

In [ ]:
# 5) Two-column features (ID, name) – here both from var_names
pd.DataFrame({"gene_id": adata.var_names, "gene_name": adata.var_names}).to_csv(
    "features.tsv", sep="\t", header=False, index=False
)

In [ ]:
# 6) One-column barcodes (cells)
pd.DataFrame({"cell": adata.obs_names}).to_csv(
    "barcodes.tsv", sep="\t", header=False, index=False
)

In [ ]:
# 7) Metadata (strings to be safe)
keep = ['sample_id', 'patient', 'tissue_group', 'n_genes_by_counts',
       'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes',
       '_scvi_batch', '_scvi_labels', 'leiden']
keep = [c for c in keep if c in adata.obs.columns]
meta = adata.obs[keep].copy()
for c in meta.columns:
    meta[c] = meta[c].astype(str).str.strip()
meta.to_csv("metadata.csv")

print("✅ Wrote counts.mtx (genes×cells) / features.tsv (2 cols) / barcodes.tsv / metadata.csv")

Export embeddings

In [ ]:
import pandas as pd

# make sure both exist
print(list(adata.obsm.keys()))  # should include "X_scVI" and maybe "X_umap"

pd.DataFrame(adata.obsm["X_scVI"], index=adata.obs_names).to_csv("scvi_embedding.csv")
if "X_umap" in adata.obsm:
    pd.DataFrame(adata.obsm["X_umap"], index=adata.obs_names).to_csv("umap_coords.csv")
print("✅ wrote scvi_embedding.csv and (optionally) umap_coords.csv")